In [1]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk.core as core
import nvitk.core.logger as logger
core.setup(globals())
log = logger.Logger()

import nvitk as nv
from nvitk.morphology.centerline_siphon import correct_siphon_centerlines

OUT_DIR = Path("/home/imarcoss/nvitk/notebooks/exploration/pesabrain-anatomy/checkpoints")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
tof = nv.imread(
    "/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/RESULTS/res_QVTPy/PESA15689521/eicab/TOF_resampled.nii.gz",
    backend='cpu', axes='XYZ'
)
eicab = nv.imread(
    "/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/RESULTS/res_QVTPy/PESA15689521/eicab/TOF_eICAB_CW.nii.gz",
    backend='cpu', axes='XYZ'
)
print("TOF:", tof, "\neICAB:", eicab)


TOF: Image(shape=(352, 406, 192), dtype=float32, backend=numpy, axes='XYZ', orientation='RAS', name='TOF_resampled.nii', modality=None, submodality=None, rescale_type='DV') 
eICAB: Image(shape=(352, 406, 192), dtype=float32, backend=numpy, axes='XYZ', orientation='RAS', name='TOF_eICAB_CW.nii', modality=None, submodality=None, rescale_type='DV')


In [3]:
tof.affine

array([[   0.5       ,   -0.        ,   -0.        ,  -78.7522583 ],
       [  -0.        ,    0.5       ,   -0.        , -107.32932281],
       [   0.        ,    0.        ,    0.5       ,  -22.17021942],
       [   0.        ,    0.        ,    0.        ,    1.        ]])

In [4]:
res = correct_siphon_centerlines(
    tof, eicab,
    correction_ids=(1, 2),
    out_dir=OUT_DIR,
    save_qc=True,
)

16:01:08 | INFO     |   ▸ correct_siphon_centerlines: shape=(352, 406, 192) correction_ids=(1, 2)


16:01:08 | INFO     |   ▸ labels present: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 16, 17] | siphon-corrected: [1, 2] | default: [3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 16, 17]
16:01:08 | INFO     |   ▸ === Seed centerlines from vessel_mask ===
16:01:11 | INFO     |   ▸ seed centerlines done in 3.09s (15 labels)
16:01:11 | INFO     |   ▸ === Default centerlines for non-ICA labels ===
16:01:13 | INFO     |   ▸ === ICA Otsu + repair + siphon centerlines ===
16:01:13 | INFO     |   ▸ --- LICA (id=1) ---
16:01:14 | INFO     |   ▸ [LICA] Otsu+erode in 0.17s: thr=58736.22604381132 vox_pre=6201 → vox_post=2127 (erode_iters=2) bbox=(144, 169, 228, 405, 35, 76)
16:01:15 | INFO     |   ▸ [LICA] eroded (iters=2): voxels=2127 β₀=1 χ=1.0 β₁=0 skel_cycles=0 suspect=False
16:01:15 | INFO     | [LICA] erosion alone cleared topology → no cut needed
16:01:15 | INFO     |   ▸ [LICA] repaired: voxels=2127 β₁=0 skel_cycles=0 suspect=False
16:01:16 | INFO     |   ▸ [LICA] no skeleton cycles → no pruning 


ICA      vox_o   vox_e   vox_r   β₁ o→e→r    cyc o→e→r  CL_pts                           action
--------------------------------------------------------------------------------------------------------------
LICA      6201    2127    2127      0→0→0        1→0→0      77  skipped (erosion alone cleared)
RICA      5785    1834    1834      1→0→0        3→0→0      82  skipped (erosion alone cleared)


In [5]:
res.get('details')

{1: {'label': 1,
  'label_name': 'LICA',
  'n_skel': 77,
  'n_skel_pruned': 77,
  'n_bridge': 0,
  'n_pts': 77,
  'base': [155, 235, 39],
  'tip': [152, 245, 70],
  'cycles': [],
  'bridge_voxels': [],
  'warning': None,
  'prep': {'otsu_info': {'label': 'LICA',
    'otsu_thresh': 58736.22604381132,
    'bbox': (144, 169, 228, 405, 35, 76),
    'n_voxels': 2127,
    'n_voxels_pre_erode': 6201,
    'erode_iters': 2},
   'otsu_mask': array([[[False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           ...,
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False]],
   
          [[False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
           [False, False, False, ..., False, False, False],
      